# TensorFly on Colab (paid-GPU workflow)

Practical end-to-end notebook for **MaleCNS-scale connectome inference + Qwen**.

**Two modes (kept separate):**

- **Benchmark mode** — timed `sim.step()` runs (`run_benchmark`). Measures latency/throughput.
- **Viewer / replay mode** — snapshot-driven `ReplayRecorder` export (`viewer/tensorfly_replay.json`) for the standalone Three.js viewer. No timing inside replay capture.

**Honesty rule:** without a real `MALECNS_EDGE_SOURCE` file this repo builds a clearly-flagged **synthetic scaffold** (`is_synthetic=True`, `synthetic-scaffold (NOT real MaleCNS v1.0)`). Nothing synthetic is presented as real MaleCNS v1.0.

Run cells top-to-bottom. Heavy full-target cells are **gated off** by default.

## 0. Install (run first)

Clone the repo, then install the `inference` extra. Restart the Colab runtime if torch/transformers were freshly installed.

In [ ]:
# In Colab, adjust the path to where you cloned this repo:
# !git clone https://github.com/<you>/fly-inference-optimizer.git
# %cd fly-inference-optimizer

!pip install -e .[inference]

## 1. Startup: runtime + GPU + model

Imports `tensorfly`, calls `get_runtime()`, resolves the Qwen model for the detected profile, and prints GPU name, VRAM, system RAM, CUDA version, PyTorch version, profile, and model.

In [ ]:
import tensorfly
from tensorfly import get_runtime, select_qwen_model

rt = get_runtime()
model_id = select_qwen_model(rt.profile.name, rt.gpu.vram_gb)

# One-block summary (GPU, VRAM, RAM, CUDA, torch, profile, model, device):
rt.print_startup_summary(model=model_id)

print()
print(f"GPU name       : {rt.gpu.name}")
print(f"GPU present    : {rt.gpu.present}")
print(f"VRAM (GB)      : {rt.gpu.vram_gb if rt.gpu.vram_gb is not None else 'n/a'}")
print(f"System RAM (GB): {rt.system.ram_gb if rt.system.ram_gb is not None else 'n/a'} (cpus={rt.system.cpu_count})")
print(f"CUDA version   : {rt.gpu.cuda_version or 'n/a'} (torch_cuda_available={rt.gpu.torch_cuda_available})")
print(f"PyTorch version: {rt.gpu.torch_version or 'n/a (torch not installed)'}")
print(f"Profile        : {rt.profile.name} (dtype={rt.profile.dtype}, batch_size={rt.profile.batch_size})")
print(f"Model          : {model_id}")
print(f"Device         : {rt.device}")

## 2. Configuration (paid Colab profiles + explicit fallback)

| Colab GPU | TensorFly profile | dtype | batch_size | Qwen model |
|---|---|---|---|---|
| A100 40/80GB | `A100` | bfloat16 | 1024 | `Qwen/Qwen3.5-9B` |
| L4 24GB | `L4` | float16 | 512 | `Qwen/Qwen3.5-9B` |
| T4 16GB | `T4` | float16 | 256 | `Qwen/Qwen3.5-4B` (fallback notice) |
| CPU / unknown | `CPU` | float32 | 64 | `Qwen/Qwen3.5-4B` (fallback notice) |

Fallback is **explicit and printed**: unknown CUDA GPU -> `T4` profile; no GPU -> `CPU` profile; insufficient VRAM for 9B (`< 20 GB`) or any non-A100/L4 profile -> `Qwen3.5-4B` with the exact line `TensorFly fallback: Qwen3.5-4B because current GPU memory is insufficient for the 9B benchmark profile.`

In [ ]:
from tensorfly import PROFILES, InferenceConfig, QwenInference, resolve_dtype, resolve_device

for name, p in PROFILES.items():
    print(f"{name:4s} dtype={p.dtype:9s} batch={p.batch_size:4d} workers={p.num_workers} chunk={p.max_neurons_per_chunk:6d}  # {p.description}")

print()
print("resolved dtype :", resolve_dtype(rt.profile.name))
print("resolved device:", resolve_device())

# Metadata only — does NOT download or load the model:
inf = QwenInference(profile_name=rt.profile.name, vram_gb=rt.gpu.vram_gb)
print(inf.metadata)
print()
print("To load + generate (downloads weights, needs GPU/RAM):")
print("  inf.load()")
print("  inf.generate_with_metrics('describe the optic lobe response')")

## 3. Mode separation

- **Benchmark mode (section 4):** small smoke benchmark with `run_benchmark`. Only the `sim.step()` loop is timed; graph build is excluded. Use it to compare configs.
- **Viewer / replay mode (section 6):** step a simulation to get **real simulation-derived activity**, record it with `ReplayRecorder`, save `viewer/tensorfly_replay.json`. Never mix timing loops into replay capture.
- **Full-target build (section 5):** constructs the 166,700-neuron / 25.6M-edge graph with cache + memmap. It is **defined but not executed** unless you flip `RUN_FULL_TARGET = True`.

## 4. Benchmark mode: small smoke benchmark (safe on any GPU/CPU)

Runs fast (< 1 min). This is the only benchmark that executes automatically.

In [ ]:
from tensorfly import MaleCNSSimulation, SimulationConfig, run_benchmark

smoke = MaleCNSSimulation(SimulationConfig(num_neurons=5_000, num_edges=20_000, seed=0))
smoke.build()
print(smoke.summary())
assert smoke.is_synthetic, "smoke run without a real source must stay flagged synthetic"

result = run_benchmark(smoke, steps=5, repeats=2, label=rt.profile.name, runtime=rt)
print(f"ms/step mean={result.ms_per_step_mean:.3f} median={result.ms_per_step_median:.3f} steps/s={result.steps_per_second:.2f} spikes={result.total_spikes}")
result.save("smoke_benchmark.json")
print("saved smoke_benchmark.json")

## 5. Full-target simulation: real MaleCNS source (gated, does NOT run automatically)

Target: **166,700 neurons / 25,600,000 edges** (MaleCNS v1.0). Set `MALECNS_EDGE_SOURCE` to a real `.npz` CSR dump (`row_ptr`/`col_idx`/`weights`) or `.csv` edgelist (`source,target[,weight]`). Without it, `build()` produces a synthetic scaffold and says so — it never claims to be real MaleCNS data.

In [ ]:
import os
from pathlib import Path
from tensorfly import N_MALECNS_EDGES, N_MALECNS_NEURONS

# Optional: point at a real MaleCNS export you uploaded to Colab / Drive.
# %env MALECNS_EDGE_SOURCE=/content/drive/MyDrive/malecns_edges.npz
edge_source = os.environ.get("MALECNS_EDGE_SOURCE", "")
print("MALECNS_EDGE_SOURCE:", edge_source or "(unset -> synthetic scaffold, honestly flagged)" )
print(f"Full target: {N_MALECNS_NEURONS:,} neurons / {N_MALECNS_EDGES:,} edges")

RUN_FULL_TARGET = False  # flip to True on a paid high-RAM runtime to actually build/run this

if RUN_FULL_TARGET:
    from tensorfly import MaleCNSSimulation, SimulationConfig

    cfg = SimulationConfig(
        num_neurons=N_MALECNS_NEURONS,
        num_edges=N_MALECNS_EDGES,
        seed=0,
        cache_dir=Path("cache/malecns"),
        use_memmap=True,  # CSR arrays live on disk, paged in as needed
    )
    full = MaleCNSSimulation(cfg, edge_source=edge_source or None)
    full.build()
    print(full.summary())
    full.save_snapshot("cache/malecns_full_summary.json")
    # Deliberately NO run_benchmark(full, ...) here: time it manually in its own cell
    # once you have confirmed free RAM/VRAM, e.g. run_benchmark(full, steps=3, repeats=2, runtime=rt).
else:
    print("Skipped full-target build (RUN_FULL_TARGET=False). Smoke benchmark above is the only auto-run workload.")

## 6. Viewer / replay mode: real simulation-derived activity -> `viewer/tensorfly_replay.json`

Steps a **small real simulation**, slices its normalized voltage/activity into `sensory | dopamine | controller` populations, and records genuine snapshots (no invented activity). The output validates against the `fly-cns-replay/1` schema documented in `viewer/README.md`.

In [ ]:
import numpy as np
from tensorfly import MaleCNSSimulation, SimulationConfig, ReplayRecorder

N_SENSORY, N_DOPA, N_CTRL = 256, 64, 128  # small but real populations for the viewer
N_TOTAL = N_SENSORY + N_DOPA + N_CTRL

sim = MaleCNSSimulation(SimulationConfig(num_neurons=N_TOTAL, num_edges=N_TOTAL * 8, seed=1))
sim.build()
print("data_source:", sim.data_source, "| is_synthetic:", sim.is_synthetic)

rec = ReplayRecorder(
    model=model_id,
    prompt="describe the optic lobe response",
    populations={"sensory": N_SENSORY, "dopamine": N_DOPA, "controller": N_CTRL},
    config={"id": rt.profile.name, "dtype": rt.profile.dtype, "batch_size": rt.profile.batch_size},
    is_synthetic=sim.is_synthetic,
    data_source=sim.data_source,
)

def norm01(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float64)
    lo, hi = float(v.min()), float(v.max())
    if hi <= lo:
        return np.zeros_like(v)
    return (v - lo) / (hi - lo)

N_FRAMES = 24
for i in range(N_FRAMES):
    spikes = sim.step()  # real LIF dynamics; returns this step's spike count
    state = norm01(sim.voltage) * 0.7 + sim.activity.astype(np.float64) * 0.3  # [0,1]-ish measured state
    sensory = state[:N_SENSORY]
    dopamine = state[N_SENSORY:N_SENSORY + N_DOPA]
    controller = state[N_SENSORY + N_DOPA:]
    rec.record_snapshot(
        t=float(i) * 0.25,
        sensory=sensory,
        dopamine=dopamine,
        controller=controller,
        trial=0,
        tokens=int(spikes),
        throughput_tps=float(spikes) / 0.25,
        reward=float(np.mean(state)),
        config_id=rt.profile.name,
    )

rec.record_event("trial_end", f"smoke replay: {N_FRAMES} measured frames ({rt.profile.name}/{model_id})")
out = rec.save("viewer/tensorfly_replay.json")
print(f"saved {out} with {len(rec)} measured frames; best={rec.best()}")
print("Populations:", rec.populations)

## 7. Serve the viewer + video output workflow

`viewer/` is standalone (no build step) but **must be served over HTTP** (ES modules + `canvas.captureStream` fail on `file://`).

**Local / Colab flow:**

```bash
# from the repo root
python -m http.server 8000 --directory viewer
# open http://localhost:8000/?replay=./tensorfly_replay.json
```

In Colab, expose port 8000 (e.g. via port-forwarding) or download `viewer/tensorfly_replay.json` and open it locally with the command above, or via Load-file / `?replay=` URL. Without a replay the header shows `SAMPLE SCAFFOLD · not measured`; with this file it shows `LIVE REPLAY · measured snapshots`.

**Video:** press **Record demo** in the viewer (captures `canvas.captureStream(60)` + ~4 Hz state JSON), then **Stop** and use **Download Video** (`.webm`) / **Download JSON** (reloadable `fly-cns-replay/1`). Optional MP4 hook: set the endpoint field (or `window.__MP4_CONVERT_URL__`) — the viewer POSTs the WebM and downloads the MP4 response, keeping WebM as fallback. See `viewer/README.md` for the full schema and controls.

In [ ]:
import json
from pathlib import Path

p = Path("viewer/tensorfly_replay.json")
assert p.exists(), "run section 6 first to generate viewer/tensorfly_replay.json"
payload = json.loads(p.read_text())
assert payload["schema"] == "fly-cns-replay/1", payload["schema"]
assert len(payload["frames"]) > 0
print(f"OK: {p} schema={payload['schema']} frames={len(payload['frames'])} model={payload['meta']['model']}")
print()
print("Serve with (run in a terminal, then open the URL):")
print("  python -m http.server 8000 --directory viewer")
print("  http://localhost:8000/?replay=./tensorfly_replay.json")